In this first step data are imported and visualized.

# Heading 1
## Heading 2
### Heading 3
*Italic* or _Italic_

**Bold** or __Bold__

~~Strikethrough~~


In [23]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from scipy import stats

# Load dataset
df = pd.read_csv("MyDatasetToClean.csv")
# print(df)
print(df.iloc[1]) #Print second line 
print(df.columns)


ID                                                          1
SellDate                                           2024-03-02
PayMethod                                                loan
PriceSold                                               18000
featuresIncluded      ['morepower', 'coating', 'turbocharge']
FinishPayDate                                      2088-06-24
BuyerAge                                                 23.0
CustomerFeedback              I'll recommend it to my friends
ProductColor                                           Iellow
ProductCategory                               Home Appliances
BuyerGender                                              Male
BuyerSalary                                               NaN
MonthlyInstallment                                        NaN
Name: 1, dtype: object
Index(['ID', 'SellDate', 'PayMethod', 'PriceSold', 'featuresIncluded',
       'FinishPayDate', 'BuyerAge', 'CustomerFeedback', 'ProductColor',
       'ProductCategory', 'B

In [24]:
# dfmissing = df.isnull().sum()   # Number of missing values per column
# for v in df.columns:
#     print(df[v].isnull())    # Rows with missing values in 'col'
print(len(df))
df['BuyerAge'].fillna(df['BuyerAge'].mean(), inplace=True)
print(df['BuyerAge'])


20
0     45.0
1     23.0
2     67.0
3     32.0
4     44.0
5     42.0
6     38.0
7     22.0
8      4.0
9     39.0
10    40.0
11    49.0
12    68.0
13    43.0
14    39.0
15    43.0
16    30.0
17    35.0
18    35.0
19    42.0
Name: BuyerAge, dtype: float64


C:\Users\39346\AppData\Local\Temp\ipykernel_4180\3797626666.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['BuyerAge'].fillna(df['BuyerAge'].mean(), inplace=True)


Removing duplicates

In [ ]:
# Drop duplicates
# Ignore the 'ID' column when checking for duplicates
dup_rows = df[df.duplicated(subset=df.columns.difference(['ID']))]

print(dup_rows)

    ID    SellDate PayMethod  PriceSold              featuresIncluded  \
18  17         NaN      card      16000  ['morepower', 'turbocharge']   
19   5  05-05-2024      cash       8000                           NaN   

   FinishPayDate  BuyerAge      CustomerFeedback ProductColor ProductCategory  \
18    2062-07-16      35.0   This product is ok.      greeen,       Furniture   
19    2040-08-10      42.0  I would buy it again   Is Orange!     Electronics   

   BuyerGender  BuyerSalary  MonthlyInstallment  
18       Other      70093.0                 NaN  
19      Female     105813.0                 NaN  


Fixing incosistent label

In [41]:
import emoji
import re
from rapidfuzz import process

# Function to clean a string
def remove_special_characters(text):
    return re.sub(r'[^A-Za-z0-9\s]', '', str(text))

def remove_stopwords(text):
    if not isinstance(text, str):
        return text
    return ' '.join(word for word in text.split() if word.lower() not in stopwords)


def remove_emoji(text):
    return emoji.replace_emoji(text, replace='')

df['ProductColor'] = df['ProductColor'].apply(remove_special_characters)
df['ProductColor'] = df['ProductColor'].astype(str).apply(remove_emoji)

stopwords = {'is', 'the', 'a', 'an', 'of', 'in'}

df['ProductColor'] = df['ProductColor'].apply(remove_stopwords)
df['ProductColor'] = df['ProductColor'].astype(str).str.strip().str.lower()




# Get unique values
unique_values = df['ProductColor'].unique()

# Choose reference labels
known_labels = ['red', 'green', 'blue', 'yellow','white','purple','orange','black']

# Function to map each value to its best match
def correct_label(val):
    match, score, _ = process.extractOne(val, known_labels)
    return match if score > 75 else val  # Only replace if confidence is high

df['ProductColor'] = df['ProductColor'].apply(correct_label)
print(df['ProductColor'])




#Print unique values
print(df['ProductColor'].unique())

0     yellow
1     yellow
2     yellow
3      black
4      green
5     orange
6       blue
7     purple
8     purple
9     purple
10    yellow
11    yellow
12    yellow
13     white
14    yellow
15      blue
16       nan
17     green
18     green
19    orange
Name: ProductColor, dtype: object
['yellow' 'black' 'green' 'orange' 'blue' 'purple' 'white' 'nan']
